# Tech Challenge Fase 2

## Notebook 01 — Bronze Orquestrador

**Projeto:** Pipeline de Dados para Análise da Alfabetização no Brasil  
**Camada:** Bronze  
**Ambiente:** AWS S3 + Databricks + Unity Catalog + External Volume  
**Notebook anterior:** `00_setup_ambiente`  
**Próximos notebooks:**

- `01_1_bronze_alunos`
- `01_2_bronze_estados`
- `01_3_bronze_municipios`
- `01_4_bronze_metas_municipios`
- `01_5_bronze_metas_ufs`

---

## Objetivo do notebook

Este notebook atua como **orquestrador lógico da camada Bronze**. Sua função é centralizar os metadados operacionais das bases recebidas, validar se os arquivos esperados estão disponíveis na camada `raw` e registrar a configuração que será utilizada pelos notebooks filhos de ingestão.

Nesta etapa, **não realizamos transformação de negócio**. A responsabilidade deste notebook é preparar a execução da ingestão Bronze de forma organizada, auditável e orientada por metadados.


## 1. Papel deste notebook na Arquitetura Medalhão

A Arquitetura Medalhão organiza os dados em camadas progressivas de maturidade:

```text
Raw → Bronze → Silver → Gold → Power BI / Machine Learning
```

Neste notebook, estamos preparando a ingestão para a camada **Bronze**, que tem como objetivo preservar os dados o mais próximo possível da origem.

### Responsabilidades deste notebook

- Ler o `config.json` criado no notebook de setup.
- Definir a tabela de metadados dos arquivos de entrada.
- Validar a existência dos arquivos esperados na camada `raw`.
- Salvar a tabela `bronze_metadata` na área de configuração do projeto.
- Gerar logs de validação dos arquivos.
- Preparar os notebooks filhos para ingestão por dataset.

### O que este notebook não faz

- Não cria novamente a estrutura completa do Data Lake.
- Não move arquivos entre pastas.
- Não aplica limpeza ou transformação de negócio.
- Não gera tabelas Silver ou Gold.

Essas responsabilidades já estão separadas em outros notebooks, mantendo o projeto modular e aderente a boas práticas de Engenharia de Dados.


## 2. Importação das bibliotecas

Nesta etapa são importadas as bibliotecas necessárias para leitura de configuração, manipulação de datas, criação de DataFrames Spark e definição explícita de schemas.

O uso de **schema explícito** é importante no Databricks Serverless porque evita erros de inferência, especialmente quando colunas de log podem conter valores nulos ou vazios.


In [0]:
from datetime import datetime
import json

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType
)


## 3. Leitura do arquivo de configuração do projeto

O arquivo `config.json` foi criado no notebook `00_setup_ambiente` e centraliza todos os caminhos do projeto.

Essa abordagem evita caminhos fixos espalhados pelos notebooks e facilita futuras mudanças de infraestrutura, como alteração de bucket, volume ou diretório base.

### Caminho esperado do projeto

```text
/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json
```

Esse caminho corresponde fisicamente ao prefixo no S3:

```text
s3://s3tc2/projetos/fiap/tech_challenge_fase2/
```


In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

BASE_PATH = config["environment"]["base_path"]
RAW_PATH = config["paths"]["raw_path"]
BRONZE_PATH = config["paths"]["bronze_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH:", RAW_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("LOG_PATH:", LOG_PATH)
print("CONFIG_PATH:", CONFIG_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)


## 4. Definição dos datasets e anos suportados

Nesta etapa definimos os domínios de dados que compõem a camada Bronze.

Os datasets foram separados por tipo de informação:

- `alunos`: microdados dos estudantes avaliados.
- `estados`: indicadores agregados por UF.
- `municipios`: indicadores agregados por município.
- `metas_municipios`: metas de alfabetização por município.
- `metas_ufs`: metas de alfabetização por UF e Brasil.

Os anos inicialmente suportados são 2023, 2024 e 2025. Essa lista pode ser expandida futuramente sem alterar a lógica principal do pipeline.


In [0]:
datasets = [
    "alunos",
    "estados",
    "municipios",
    "metas_municipios",
    "metas_ufs"
]

anos = [2023, 2024, 2025]

print("Datasets suportados:", datasets)
print("Anos suportados:", anos)


## 5. Construção da tabela de metadados Bronze

A ingestão Bronze será orientada por uma tabela de metadados. Cada linha representa um arquivo esperado e contém informações necessárias para o respectivo notebook filho realizar a ingestão.

### Informações registradas

- Nome do dataset.
- Ano de referência.
- Nome físico do arquivo.
- Caminho na camada `raw`.
- Caminho de destino na camada `bronze`.
- Formato do arquivo (`csv` ou `excel`).
- Separador e encoding, quando aplicável.
- Nome da aba Excel, quando aplicável.
- Linha inicial a ser ignorada em planilhas com cabeçalho duplo.
- Coluna que representa o ano.
- Notebook responsável pela ingestão.

Essa abordagem reduz duplicidade de código e facilita a inclusão de novos arquivos no futuro.


In [0]:
bronze_metadata = []

# =====================================================
# ALUNOS
# =====================================================
for ano in anos:
    bronze_metadata.append({
        "dataset": "alunos",
        "ano": ano,
        "file_name": f"TS_ALUNO_{ano}.csv",
        "raw_path": f"{RAW_PATH}/alunos/TS_ALUNO_{ano}.csv",
        "bronze_path": f"{BRONZE_PATH}/alunos/ano={ano}",
        "source_format": "csv",
        "separator": ";",
        "encoding": "ISO-8859-1",
        "header": True,
        "sheet_name": "",
        "skip_rows": 0,
        "ano_column": "NU_ANO_AVALIACAO",
        "notebook": "01_1_bronze_alunos",
        "active": True
    })

# =====================================================
# ESTADOS
# =====================================================
for ano in anos:
    bronze_metadata.append({
        "dataset": "estados",
        "ano": ano,
        "file_name": f"TS_ESTADO_{ano}.csv",
        "raw_path": f"{RAW_PATH}/estados/TS_ESTADO_{ano}.csv",
        "bronze_path": f"{BRONZE_PATH}/estados/ano={ano}",
        "source_format": "csv",
        "separator": ";",
        "encoding": "ISO-8859-1",
        "header": True,
        "sheet_name": "",
        "skip_rows": 0,
        "ano_column": "NU_ANO_AVALIACAO",
        "notebook": "01_2_bronze_estados",
        "active": True
    })

# =====================================================
# MUNICÍPIOS
# =====================================================
for ano in anos:
    bronze_metadata.append({
        "dataset": "municipios",
        "ano": ano,
        "file_name": f"TS_MUNICIPIO_{ano}.csv",
        "raw_path": f"{RAW_PATH}/municipios/TS_MUNICIPIO_{ano}.csv",
        "bronze_path": f"{BRONZE_PATH}/municipios/ano={ano}",
        "source_format": "csv",
        "separator": ";",
        "encoding": "ISO-8859-1",
        "header": True,
        "sheet_name": "",
        "skip_rows": 0,
        "ano_column": "NU_ANO_AVALIACAO",
        "notebook": "01_3_bronze_municipios",
        "active": True
    })

# =====================================================
# METAS MUNICÍPIOS
# =====================================================
for ano in anos:
    bronze_metadata.append({
        "dataset": "metas_municipios",
        "ano": ano,
        "file_name": f"metas_municipios_{ano}.xlsx",
        "raw_path": f"{RAW_PATH}/metas_municipios/metas_municipios_{ano}.xlsx",
        "bronze_path": f"{BRONZE_PATH}/metas_municipios/ano={ano}",
        "source_format": "excel",
        "separator": "",
        "encoding": "",
        "header": True,
        "sheet_name": "Divulgação Alfabet Municipio",
        "skip_rows": 1,
        "ano_column": "ANO",
        "notebook": "01_4_bronze_metas_municipios",
        "active": True
    })

# =====================================================
# METAS UFS
# =====================================================
for ano in anos:
    bronze_metadata.append({
        "dataset": "metas_ufs",
        "ano": ano,
        "file_name": f"metas_ufs_{ano}.xlsx",
        "raw_path": f"{RAW_PATH}/metas_ufs/metas_ufs_{ano}.xlsx",
        "bronze_path": f"{BRONZE_PATH}/metas_ufs/ano={ano}",
        "source_format": "excel",
        "separator": "",
        "encoding": "",
        "header": True,
        "sheet_name": "Divulgação Alfabet UF e Brasil",
        "skip_rows": 1,
        "ano_column": "ANO",
        "notebook": "01_5_bronze_metas_ufs",
        "active": True
    })

print(f"Total de arquivos registrados nos metadados Bronze: {len(bronze_metadata)}")


## 6. Criação do DataFrame de metadados com schema explícito

O uso de schema explícito garante maior estabilidade no Databricks Serverless.

Alguns campos, como `sheet_name`, `separator` e `encoding`, podem não ser aplicáveis a todos os formatos. Em vez de utilizar valores nulos, registramos string vazia ou zero, evitando erro de inferência de tipos.


In [0]:
schema_bronze_metadata = StructType([
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("file_name", StringType(), True),
    StructField("raw_path", StringType(), True),
    StructField("bronze_path", StringType(), True),
    StructField("source_format", StringType(), True),
    StructField("separator", StringType(), True),
    StructField("encoding", StringType(), True),
    StructField("header", BooleanType(), True),
    StructField("sheet_name", StringType(), True),
    StructField("skip_rows", IntegerType(), True),
    StructField("ano_column", StringType(), True),
    StructField("notebook", StringType(), True),
    StructField("active", BooleanType(), True)
])

df_bronze_metadata = spark.createDataFrame(
    bronze_metadata,
    schema=schema_bronze_metadata
)

display(df_bronze_metadata.orderBy("dataset", "ano"))


## 7. Persistência da tabela de metadados Bronze

Os metadados serão salvos na área de configuração do projeto, em formato Parquet.

Essa tabela será lida pelos notebooks filhos para identificar automaticamente quais arquivos devem ser processados para cada dataset.

### Saída esperada

```text
config/bronze_metadata/
```

Optamos por `coalesce(1)` porque a tabela de metadados é pequena e deve gerar apenas um arquivo Parquet, evitando o problema de pequenos arquivos em excesso.


In [0]:
metadata_path = f"{CONFIG_PATH}/bronze_metadata"

(
    df_bronze_metadata
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(metadata_path)
)

print("Tabela bronze_metadata salva em:")
print(metadata_path)


## 8. Validação cruzada dos arquivos esperados na camada Raw

O notebook de setup já realiza uma validação inicial dos arquivos. Aqui fazemos uma **validação cruzada com a tabela de metadados**, garantindo que cada arquivo registrado para ingestão realmente existe no caminho informado.

Essa etapa evita que os notebooks filhos iniciem execução com arquivos ausentes, nomes incorretos ou upload realizado em pasta errada.


In [0]:
validacao_arquivos = []

for item in bronze_metadata:
    raw_path = item["raw_path"]

    try:
        dbutils.fs.ls(raw_path)
        status = "OK"
        erro = ""
    except Exception as e:
        status = "PENDENTE"
        erro = str(e)

    validacao_arquivos.append({
        "dataset": str(item["dataset"]),
        "ano": int(item["ano"]),
        "file_name": str(item["file_name"]),
        "raw_path": str(raw_path),
        "status": str(status),
        "erro": str(erro),
        "execution_date": str(EXECUTION_DATE)
    })

schema_validacao = StructType([
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("file_name", StringType(), True),
    StructField("raw_path", StringType(), True),
    StructField("status", StringType(), True),
    StructField("erro", StringType(), True),
    StructField("execution_date", StringType(), True)
])

df_validacao_arquivos = spark.createDataFrame(
    validacao_arquivos,
    schema=schema_validacao
)

display(df_validacao_arquivos.orderBy("dataset", "ano"))


## 9. Persistência do log de validação

O resultado da validação cruzada é salvo na camada de logs.

Esses registros permitem auditoria e rastreabilidade operacional, além de facilitar análise de falhas quando algum arquivo obrigatório não é encontrado.

### Saída esperada

```text
logs/pipeline_execution/bronze/validacao_metadata_execution_date=YYYY-MM-DD/
```


In [0]:
validation_path = f"{LOG_PATH}/pipeline_execution/bronze/validacao_metadata_execution_date={EXECUTION_DATE}"

(
    df_validacao_arquivos
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(validation_path)
)

print("Log de validação dos metadados salvo em:")
print(validation_path)


## 10. Resumo executivo da validação

Nesta etapa consolidamos o status dos arquivos por dataset.

Esse resumo indica quantos arquivos eram esperados, quantos foram encontrados e se existe alguma pendência que possa impedir a execução dos notebooks filhos.


In [0]:
df_resumo_validacao = (
    df_validacao_arquivos
    .groupBy("dataset")
    .agg(
        F.count("*").alias("arquivos_esperados"),
        F.sum(F.when(F.col("status") == "OK", 1).otherwise(0)).alias("arquivos_encontrados"),
        F.sum(F.when(F.col("status") == "PENDENTE", 1).otherwise(0)).alias("arquivos_pendentes")
    )
    .withColumn(
        "status_dataset",
        F.when(F.col("arquivos_pendentes") == 0, "OK").otherwise("PENDENTE")
    )
    .orderBy("dataset")
)

display(df_resumo_validacao)


## 11. Checklist final do orquestrador Bronze

O checklist final indica se o catálogo operacional da Bronze está pronto para ser utilizado pelos notebooks filhos.

Em uma execução final do projeto, é recomendado interromper o fluxo caso existam arquivos pendentes. Durante testes, esse bloqueio pode ser mantido desativado ou comentado.


In [0]:
arquivos_pendentes = (
    df_validacao_arquivos
    .filter(F.col("status") == "PENDENTE")
    .count()
)

if arquivos_pendentes > 0:
    print(f"ATENÇÃO: existem {arquivos_pendentes} arquivos pendentes na camada raw.")
    print("Revise os uploads antes de executar os notebooks filhos da Bronze.")
else:
    print("Todos os arquivos registrados nos metadados foram encontrados.")
    print("Orquestrador Bronze concluído com sucesso.")


## 12. Resultado esperado

Ao final deste notebook, espera-se que:

- A configuração do projeto tenha sido carregada corretamente.
- A tabela `bronze_metadata` tenha sido criada e persistida.
- Todos os arquivos esperados tenham sido validados na camada `raw`.
- Os logs de validação tenham sido salvos.
- O ambiente esteja pronto para execução dos notebooks filhos da Bronze.

### Artefatos gerados

```text
config/bronze_metadata/
logs/pipeline_execution/bronze/validacao_metadata_execution_date=YYYY-MM-DD/
```

### Próximos notebooks

- `01_1_bronze_alunos`
- `01_2_bronze_estados`
- `01_3_bronze_municipios`
- `01_4_bronze_metas_municipios`
- `01_5_bronze_metas_ufs`
